In [1]:
# Cell 1: Installations
%pip install pytorch-forecasting pytorch-lightning torch pandas numpy

  Obtaining dependency information for pytorch-forecasting from https://files.pythonhosted.org/packages/5d/19/c72a0e1e9a4972d6d01eab769fe8dd262f57a64bb8a2a25e79e636f61085/pytorch_forecasting-1.7.0-py3-none-any.whl.metadata
  Obtaining dependency information for pytorch-lightning from https://files.pythonhosted.org/packages/8b/4d/5740c27110b83634d8491c3b5facf0111b3e554c3164f4fb953be9bddaf6/pytorch_lightning-2.6.5-py3-none-any.whl.metadata
  Obtaining dependency information for lightning<2.7.0,>=2.0.0 from https://files.pythonhosted.org/packages/e7/c5/fca7144236b6fa3279d0fb3172b32576c5ad8b84a63b9432ad6592d24847/lightning-2.6.5-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/44.8 kB ? eta -:--:--
     ---------------------------------------- 44.8/44.8 kB 1.1 MB/s eta 0:00:00
  Obtaining dependency information for scikit-base<0.14.0 from https://files.pythonhosted.org/packages/d8/2e/1488716735c31bf82c6898759d12771b3e0ff5daeb0f01b73b82a4462ad9/scikit_base-0.13.2-


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2: Data Loading and Time Indexing
import pandas as pd
import numpy as np

# 1. Load the data
df = pd.read_parquet('df_train.parquet')
df['date'] = pd.to_datetime(df['date'])

# 2. Create a proper datetime column combining date and the start of the slot
# e.g., '18:00-19:00' becomes '18:00:00'
df['start_hour'] = df['slot'].apply(lambda x: x.split('-')[0])
df['datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['start_hour'])

# 3. Sort chronologically so the time index flows correctly
df = df.sort_values(['turf_id', 'datetime']).reset_index(drop=True)

# 4. Create the crucial 'time_idx' (continuous integer step per hour)
# We subtract the absolute minimum datetime and divide by 1 hour
min_datetime = df['datetime'].min()
df['time_idx'] = ((df['datetime'] - min_datetime) / pd.Timedelta(hours=1)).astype(int)

# 5. Ensure target is float (PyTorch requires floats for continuous prediction)
# If predicting binary 'is_booked', we convert to float32.
if 'is_booked' in df.columns:
    df['is_booked'] = df['is_booked'].astype('float32')

print(f"✅ Data prepared! Time Index ranges from {df['time_idx'].min()} to {df['time_idx'].max()}")
display(df[['turf_id', 'datetime', 'time_idx', 'is_booked']].head())

✅ Data prepared! Time Index ranges from 0 to 23991


,turf_id,datetime,time_idx,is_booked
0,turf_001,2023-09-24 06:00:00,0,0.0
1,turf_001,2023-09-24 07:00:00,1,0.0
2,turf_001,2023-09-24 08:00:00,2,0.0
3,turf_001,2023-09-24 09:00:00,3,0.0
4,turf_001,2023-09-24 10:00:00,4,0.0


In [ ]:
# Cell 3: Building the PyTorch Forecasting Dataset (UPDATED)
from pytorch_forecasting import TimeSeriesDataSet

# We want the model to look back 7 days (7 * 24 = 168 hours) to predict the next 24 hours.
max_encoder_length = 168 
max_prediction_length = 24  

# We convert categoricals to strings because PyTorch Forecasting expects string categories
categorical_cols = ['turf_id', 'sport_type', 'slot', 'day_of_week']
for col in categorical_cols:
    df[col] = df[col].astype(str)

# Define the TFT Dataset
training_dataset = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="is_booked", 
    group_ids=["turf_id"], # 🌟 FIX 1: Treat the whole turf as one continuous timeline
    
    min_encoder_length=max_encoder_length // 2, 
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    
    # 1. Static variables (Never change)
    static_categoricals=["turf_id", "sport_type"],
    
    # 2. Known future variables (We know these in advance)
    time_varying_known_categoricals=["slot", "day_of_week"],
    time_varying_known_reals=["time_idx", "base_hourly_charge"], 
    
    # 3. Unknown future variables (We only know the past history of these)
    time_varying_unknown_reals=["is_booked"], 
    
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True # 🌟 FIX 2: Tell PyTorch to handle the overnight closures
)

print("✅ TFT TimeSeriesDataSet successfully constructed!")

f:\FT\ml\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
# Cell 4 & 5: DataLoaders, The Deep Learning Model, & Training Loop
import lightning.pytorch as pl  # 🌟 THE FIX: Using the correct Lightning namespace
from pytorch_forecasting.models.temporal_fusion_transformer import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch.callbacks import EarlyStopping # 🌟 Updated namespace here too

# 1. Create the Validation Dataset & DataLoaders
validation = TimeSeriesDataSet.from_dataset(
    training_dataset, 
    df, 
    predict=True, 
    stop_randomization=True
)

batch_size = 64  
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

# 2. Initialize the TFT Architecture
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=16,  
    attention_head_size=1, 
    dropout=0.1, 
    hidden_continuous_size=8, 
    output_size=7,  
    loss=QuantileLoss(),
    reduce_on_plateau_patience=4,
)

print(f"🧠 Model built! Number of parameters: {tft.size()/1e3:.1f}k")

# 3. Configure the Trainer
early_stop_callback = EarlyStopping(
    monitor="val_loss", 
    min_delta=1e-4, 
    patience=5, 
    verbose=True, 
    mode="min"
)

trainer = pl.Trainer(
    max_epochs=15,
    accelerator="auto", 
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
    enable_model_summary=True
)

# 4. Train the Network!
print("🚀 Starting PyTorch Lightning Training...")
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
print("🏁 Training Complete!")